In [1]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import os
import matplotlib.pyplot as plt
import csv

# ===================== 配置（直接填好你的路径） =====================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 模型权重
WEIGHT_PATH = r"D:\KY\SPHERE-LABELLED DATASET\Train_ResNet50_SE_GAM_FINAL1\best.pth"
# 预测文件夹（val验证集文件夹）
IMG_FOLDER = r"D:\KY\SPHERE-LABELLED DATASET\CNN_Data_Arranged\320p_split\val"
# 类别映射，和训练代码保持一致
LABEL_MAP = {0: "clean", 1: "broken", 2: "dirty"}

# ===================== SE模块、GAM模块（和训练代码完全一致） =====================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels//reduction),
            nn.ReLU(True),
            nn.Linear(channels//reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b,c,_,_ = x.size()
        y = self.avg(x).view(b,c)
        y = self.fc(y).view(b,c,1,1)
        return x * y.expand_as(x)

class GAM_ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean(dim=[2,3]) + x.amax(dim=[2,3])
        y = self.fc(y).view(b,c,1,1)
        return x * y

class GAM_SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        max_out,_ = torch.max(x, dim=1, keepdim=True)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        out = torch.cat([max_out, avg_out], dim=1)
        out = self.conv(out)
        return x * self.sigmoid(out)

class GAM(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.ca = GAM_ChannelAttention(channels, reduction)
        self.sa = GAM_SpatialAttention()
    def forward(self, x):
        x = self.ca(x)
        x = self.sa(x)
        return x

# ===================== 模型定义【和训练代码一模一样】 =====================
class ResNet50_SE_GAM(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        self.se = SEBlock(2048)
        self.gam = GAM(2048)
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(2048, num_classes)
    def forward(self, x):
        x = self.backbone(x)
        x = self.se(x)
        x = self.gam(x)
        x = self.avg(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.fc(x)

# ===================== 预处理（和val/test预处理保持一致） =====================
pred_transform = transforms.Compose([
    transforms.Resize((320,320)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ===================== 加载模型 =====================
model = ResNet50_SE_GAM(num_classes=3).to(DEVICE)
model.load_state_dict(torch.load(WEIGHT_PATH, map_location=DEVICE))
model.eval()
print("✅ 模型加载成功！")

# ===================== 单张图片预测函数 =====================
def predict_single_image(img_path):
    img = Image.open(img_path).convert("RGB")
    input_img = pred_transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(input_img)
        prob = torch.softmax(logits, dim=1)
        pred_idx = torch.argmax(prob, dim=1).item()
        pred_class = LABEL_MAP[pred_idx]
        confidence = prob[0, pred_idx].item()
    return pred_class, confidence

# ===================== 批量预测val文件夹（子文件夹clean/broken/dirty） =====================
def predict_val_dataset(root_folder, csv_save_path="val_predict_result.csv"):
    result_list = []
    # 遍历三个类别子文件夹
    cls_folders = ["clean", "broken", "dirty"]
    for true_cls in cls_folders:
        cls_path = os.path.join(root_folder, true_cls)
        img_list = [os.path.join(cls_path,f) for f in os.listdir(cls_path)
                    if f.lower().endswith(("jpg","png","jpeg"))]
        print(f"\n类别文件夹 {true_cls} 共 {len(img_list)} 张图片")
        for img_file in img_list:
            pred_cls, conf = predict_single_image(img_file)
            result_list.append([os.path.basename(img_file), true_cls, pred_cls, f"{conf:.4f}"])
            print(f"图片:{os.path.basename(img_file)} | 真实标签:{true_cls} | 预测:{pred_cls} | 置信度:{conf:.4f}")
    # 保存csv，包含文件名、真实类别、预测类别、置信度
    with open(csv_save_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["filename","true_label","predict_label","confidence"])
        writer.writerows(result_list)
    print(f"\n✅ 预测结果已保存至: {csv_save_path}")
    return result_list

# ===================== 运行预测 =====================
if __name__ == "__main__":
    results = predict_val_dataset(IMG_FOLDER)


✅ 模型加载成功！

类别文件夹 clean 共 428 张图片
图片:0428dc35-000406108_1_Clean_3675_1118.png | 真实标签:clean | 预测:clean | 置信度:0.9126
图片:04c216da-09184P60181117135_Clean_1589_2666.png | 真实标签:clean | 预测:dirty | 置信度:0.6686
图片:04c216da-09184P60181117135_Clean_1615_615.png | 真实标签:clean | 预测:clean | 置信度:0.9629
图片:04c216da-09184P60181117135_Clean_2119_2658.png | 真实标签:clean | 预测:clean | 置信度:0.9337
图片:04c216da-09184P60181117135_Clean_2658_111.png | 真实标签:clean | 预测:clean | 置信度:0.9501
图片:04c216da-09184P60181117135_Clean_3666_1683.png | 真实标签:clean | 预测:clean | 置信度:0.9620
图片:04c216da-09184P60181117135_Clean_4179_2692.png | 真实标签:clean | 预测:clean | 置信度:0.9555
图片:04c216da-09184P60181117135_Clean_4179_641.png | 真实标签:clean | 预测:clean | 置信度:0.9294
图片:04c216da-09184P60181117135_Clean_4701_1145.png | 真实标签:clean | 预测:clean | 置信度:0.9725
图片:04c216da-09184P60181117135_Clean_4709_119.png | 真实标签:clean | 预测:clean | 置信度:0.9579
图片:04c216da-09184P60181117135_Clean_581_606.png | 真实标签:clean | 预测:clean | 置信度:0.9630
图片:09f76ad8-09184P6018